In [1]:
import numpy as np
from model.grid_init import initialize_grid_2d, initialize_grid_3d
from model.parameters import get_default_parameters
from model.population_growth import *
from visualization.plots import plot_heatmap_midsection
from visualization.animation import animate_heatmap_section
from ipywidgets import *
import matplotlib.pyplot as plt

In [2]:
params = get_default_parameters()
time_steps = 3000 # 30 hours

In [3]:
def simulate_perfusion(params, time_steps):
    density_grid, concentration_grid = initialize_grid_3d(lambda : np.random.rand(), params.grid_dims)
    drug_source_grid = drug_source(params.source_concentration, params.source_position, params.grid_dims)
    concentration_grid += drug_source_grid * params.dt
    density_grid_history = [density_grid.copy()]
    concentration_grid_history = [concentration_grid.copy()]

    for t in range(time_steps):
        concentration_grid = update_concentration_perfusion(concentration_grid, drug_source_grid, params.diffusion_rate, params.dt, params.dx)
        drug_effect_grid = drug_effect(concentration_grid, params.EC50, params.hill_coefficient)
        density_grid = update_density(density_grid, drug_effect_grid, params.r, params.alpha, params.dt)
        density_grid_history.append(density_grid.copy())
        concentration_grid_history.append(concentration_grid.copy())

    return density_grid_history, concentration_grid_history

In [4]:
def simulate_intervals(params, time_steps):
    density_grid, concentration_grid = initialize_grid_3d(lambda : np.random.rand(), params.grid_dims)
    concentration_grid = add_dose(concentration_grid, params.source_concentration, params.source_position)
    density_grid_history = [density_grid.copy()]
    concentration_grid_history = [concentration_grid.copy()]

    for t in range(time_steps):
        if (t * params.dt) % params.dose_interval == 0:
            concentration_grid = add_dose(concentration_grid, params.source_concentration, params.source_position)
        concentration_grid = update_concentration_3d(concentration_grid, params.diffusion_rate, params.dt, params.dx)
        drug_effect_grid = drug_effect(concentration_grid, params.EC50, params.hill_coefficient)
        density_grid = update_density(density_grid, drug_effect_grid, params.r, params.alpha, params.dt)
        density_grid_history.append(density_grid.copy())
        concentration_grid_history.append(concentration_grid.copy())

    return density_grid_history, concentration_grid_history

In [2]:
params = get_default_parameters()
time_steps = 3000 # 30 hours

density_grid, concentration_grid = initialize_grid_3d(lambda : np.random.rand(), params.grid_dims)
#drug_source_grid = drug_source(params.source_concentration, params.source_position, params.grid_dims)
concentration_grid = add_dose(concentration_grid, params.source_concentration, params.source_position)
density_grid_history = [density_grid.copy()]
concentration_grid_history = [concentration_grid.copy()]

In [3]:
for t in range(time_steps):
    if (t * params.dt) % params.dose_interval == 0:
        concentration_grid = add_dose(concentration_grid, params.source_concentration, params.source_position)
    concentration_grid = update_concentration_3d(concentration_grid, params.diffusion_rate, params.dt, params.dx)
    drug_effect_grid = drug_effect(concentration_grid, params.EC50, params.hill_coefficient)
    density_grid = update_density(density_grid, drug_effect_grid, params.r, params.alpha, params.dt)
    density_grid_history.append(density_grid.copy())
    concentration_grid_history.append(concentration_grid.copy())


In [5]:
density_grid_history_perfusion, concentration_grid_history_perfusion = simulate_perfusion(params, time_steps)
density_grid_history_intervals, concentration_grid_history_intervals = simulate_intervals(params, time_steps)

In [7]:
perfusion_history = {
    'density': density_grid_history_perfusion,
    'concentration': concentration_grid_history_perfusion
}
intervals_history = {
    'density': density_grid_history_intervals,
    'concentration': concentration_grid_history_intervals
}

In [12]:
def plotter(timestep, history):
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(history['concentration'][timestep][:, :, params.grid_dims[2]//2], cmap='hot', vmin=0, vmax=params.dose)
    plt.title(f'Concentration at Timestep {timestep}')
    plt.colorbar(label='Concentration (mM)')
    plt.subplot(1, 2, 2)
    plt.imshow(history['density'][timestep][:, :, params.grid_dims[2]//2], cmap='viridis', vmin=0, vmax=1)
    plt.title(f'Density at Timestep {timestep}')
    plt.colorbar(label='Density (normalized)')
    plt.show()

In [ ]:
interactive(plotter, timestep= Play(min=0, max=time_steps, value=0, interval=50), 
            history = fixed(perfusion_history))

interactive(children=(Play(value=0, description='timestep', interval=50, max=3000), Output()), _dom_classes=('…

In [14]:
interactive(plotter, timestep= Play(min=0, max=time_steps, value=0, interval=50), 
            history = fixed(intervals_history))

interactive(children=(Play(value=0, description='timestep', interval=50, max=3000), Output()), _dom_classes=('…